In [1]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR, FIGURES_DIR, MODELS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob
from collections import defaultdict

from models.DeepConvLSTM import DeepConvLSTM, HARDataset, collate_fn
import optuna
import optuna.visualization as vis
import train
import torch
import train_with_cm

import matplotlib.pyplot as plt
#definisco il path da cui leggere i .csv

from utils.log_config import logger
from utils.data_preprocessing import  get_class_distribution

from optuna.pruners import MedianPruner

from figures import plot_CM, combine_kfold_confusion_matrices
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from utils import mapping_activity

#definisco il path da cui leggere i .csv

path="C:\codes\HumanActivityRecognition\data\downstream_data"
logger.debug(path)

#CREO SOTTOCARTELLA NELLA CARTELLA FIGURES_DIR CHE SI CHIAMA spoon_figures_lp
figures_spoon_path = os.path.join(FIGURES_DIR, 'spoon_figures_LP')
os.makedirs(figures_spoon_path, exist_ok=True)  # Crea la cartella




#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path
# Definisco il percorso della cartella contenente i CSV

# Nome del file CSV finale
final_csv_path = os.path.join(path, 'df_SPOON_non_null.csv')

# Controllo se il file esiste già
if os.path.exists(final_csv_path):
    logger.debug(f"Il file {final_csv_path} esiste già. Lo sto caricando...")
    df_spoon = pd.read_csv(final_csv_path)
else:
    #trovo tutti i file che corrispondono a "DO" nella cartella path e li stampo a schermo
    files = glob.glob(os.path.join(path, "*_SP*.csv"))
    logger.debug(f"Files: {files}")
    
    kid_spoon, kid_spoon_no_null = [], [] # liste per salvare utenti prima e dopo il merge 
    df_list_spoon = [] # lista vuota per appendere i dataframe con attività non nulla

    for file in files:
        df = pd.read_csv(file)
        logger.debug(f"Original shape: {df.shape}")
        kid_spoon.append(df['kid_id'].unique())
        df = df[df['action_id'] != 0] #filtro le righe con action_id non nullo
        logger.debug(f"Filtered shape: {df.shape}")
        kid_spoon_no_null.append(df['kid_id'].unique())

        logger.debug(f"Columns: {df.columns}")
        logger.debug(f"Action counts:\n{df['action'].value_counts()}")
        logger.debug(f"Toy counts:\n{df['toy_id'].value_counts()}")
        logger.debug("="*50)  # Separatore tra i file
    
    # mi stampo gli utenti prima di fare il merge e dopo il merge
    if len(kid_spoon) > 0:
        logger.info(f"Numero di utenti che hanno fatto almeno un azione: {len(kid_spoon_no_null)/len(kid_spoon)}")
    else:
        logger.info("Nessun utente trovato nei file analizzati (kid_spoon è vuoto).")

    '''
    Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
    al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
    appendo tutte le righe non nulle in un unico dataframe per tutti i file che terminano in .csv nella cartella path
    '''

    for file in os.listdir(path):
        if not file.endswith('.csv'):
            continue
    
        #leggo solo i file che dopo l'undescore ha SP*.csv
        if file.split('_')[-1].startswith('SP') and file.endswith('.csv'): #controllo che il file termini con .csv
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_spoon.append(df_temp)

    df_spoon = pd.concat(df_list_spoon)
    logger.info(f"Shape finale del dataframe: {df_spoon.shape}")
    logger.info(f"Colonne del dataframe: {list(df_spoon.columns)}")
    logger.info(f"Conteggio delle azioni:\n{df_spoon['action'].value_counts()}")

    # salvo il dataframe
    df_spoon.to_csv(os.path.join(path,'df_SPOON_non_null.csv'),index=False)


#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

#filtro le classi per trovare solo quelle che mi interessano
logger.info("FILTRAGGIO CLASSI DAL DATASET SPOON")

# Mostra distribuzione originale
logger.info("Distribuzione action_id originale:")
original_counts = df_spoon['action_id'].value_counts().sort_index()
for action_id, count in original_counts.items():
    logger.info(f"   Action {action_id}: {count} righe")


# Rimuovo le classi che non mi interessano
classes_to_remove = [4]
logger.info(f"Rimozione action_id: {classes_to_remove}")

df_spoon_filtered = df_spoon[~df_spoon['action_id'].isin(classes_to_remove)] # filtro il dataframe per rimuovere le classi che non mi interessano

logger.info(f"Righe prima del filtro: {len(df_spoon)}")
logger.info(f"Righe dopo il filtro: {len(df_spoon_filtered)}")
logger.info(f"Righe rimosse: {len(df_spoon) - len(df_spoon_filtered)}")


logger.info("Distribuzione action_id dopo filtro:")
filtered_counts = df_spoon_filtered['action_id'].value_counts().sort_index()
for action_id, count in filtered_counts.items():
    logger.info(f"   Action {action_id}: {count} righe")


#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_spoon_filtered['action_id'].unique():
    df_action = df_spoon_filtered[df_spoon_filtered["action_id"] == action_id] #filtro il dataframe in base all'attività
    logger.debug(f"Dimensioni del dataframe df_spoon_action_{action_id} - {df_action.shape}") #log delle dimensioni del dataframe
    logger.info(f"Conteggio delle attività per df_spoon_action_{action_id}") #log del conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_spoon_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    logger.debug(f"Salvato il dataframe df_spoon_action_{action_id}.csv")

<>:33: SyntaxWarning: invalid escape sequence '\c'
<>:33: SyntaxWarning: invalid escape sequence '\c'
C:\Users\n.laporta.inst\AppData\Local\Temp\ipykernel_24176\2171396355.py:33: SyntaxWarning: invalid escape sequence '\c'
  path="C:\codes\HumanActivityRecognition\data\downstream_data"
2025-07-27 15:04:51,669 - INFO - myapp - Logging configured for ipykernel_launcher in C:\codes\HumanActivityRecognition\HumanActivityRecognition\logs\ipykernel_launcher.log
2025-07-27 15:04:51,678 - INFO - myapp - GPU not available, training on CPU.
c:\Users\nicolo.laporta\OneDrive - SUPSI\Desktop\virtual_envs\har_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-27 15:04:52,886 - DEBUG - matplotlib - matplotlib data path: c:\Users\nicolo.laporta\OneDrive - SUPSI\Desktop\virtual_envs\har_venv\Lib\site-packages\matplotlib\

In [6]:
def sliding_window(a, ws, ss=None, flatten=True, min_pad_samples=30, extreme_pad_samples=10, start_row_idx=0):
    '''
    Applica una finestra mobile (sliding window) su un array multidimensionale.
    Salva l'informazione relativa alla quantità di padding applicato per non perdere nessuna finestra.
    Traccia anche gli indici delle righe originali del dataframe.
    
    Nuovo parametro:
        start_row_idx - Indice di partenza delle righe nel dataframe originale per questo bambino
    
    Ritorna:
        strided - Array contenente tutte le finestre n-dimensionali estratte da `a`
        padding_code_vector - Array contenente codici di padding
        row_indices_windows - Array contenente gli indici delle righe per ogni finestra
    '''

    if ss is None:
        ss = ws
    
    ws = norm_shape(ws)
    ss = norm_shape(ss)
    ws = np.array(ws)
    ss = np.array(ss)
    shape = np.array(a.shape)
    
    ls = [len(shape), len(ws), len(ss)]
    if 1 != len(set(ls)):
        raise ValueError(f'a.shape, ws e ss devono avere la stessa lunghezza. Valori ricevuti: {ls}')
    
    padding_code_vector = []
    
    # Creo array di indici delle righe corrispondenti ai dati originali
    original_row_indices = np.arange(start_row_idx, start_row_idx + len(a))
    
    # Se la lunghezza della finestra è maggiore del numero di campioni disponibili, aggiungo padding
    if np.any(ws > shape):
        logger.debug(f"La lunghezza della finestra è maggiore della lunghezza dell'array. Applico padding.")
        num_actual_samples = len(a)
        if num_actual_samples < extreme_pad_samples:
            logger.warning("Non ci sono abbastanza campioni per creare una finestra valida. Campioni scartati.")
            return np.array([]), np.array([]), np.array([])
        else:
            num_padding = ws[0] - num_actual_samples
            padding_start = np.random.randint(1, num_padding) if num_padding > 1 else 1
            padding_end = num_padding - padding_start
            
            logger.info(f"Padding iniziale: {padding_start}, Padding finale: {padding_end}")
            
            # Aggiungo padding ai dati
            padded_samples = np.concatenate((np.zeros((padding_start,) + a.shape[1:]), a), axis=0)
            padded_samples = np.concatenate((padded_samples, np.zeros((padding_end,) + a.shape[1:])), axis=0)
            a = padded_samples
            
            # Aggiungo padding agli indici delle righe (-1 per indicare padding)
            padding_start_indices = np.full(padding_start, -1)
            padding_end_indices = np.full(padding_end, -1)
            original_row_indices = np.concatenate((padding_start_indices, original_row_indices, padding_end_indices))
            
            if num_actual_samples < min_pad_samples and num_actual_samples > extreme_pad_samples:
                padding_code_vector.append(2)
            elif num_actual_samples < ws[0]:
                padding_code_vector.append(1)
                
            logger.info(f"Nuova lunghezza array: {len(a)}")
            shape = np.array(a.shape)

    # Calcolo il numero di finestre
    newshape = norm_shape(((shape - ws) // ss) + 1)
    newshape += norm_shape(ws)
    newstrides = norm_shape(np.array(a.strides) * ss) + a.strides
    
    # Creo l'array strided per i dati
    strided = ast(a, shape=newshape, strides=newstrides)
    
    # Creo l'array strided per gli indici delle righe
    row_indices_strided = ast(original_row_indices, 
                            shape=(newshape[0], ws[0]), 
                            strides=(original_row_indices.strides[0] * ss[0], original_row_indices.strides[0]))
    
    logger.info(f"Numero di finestre estratte: {strided.shape[0]}")

    # Gestisco i campioni rimanenti
    remaining_samples = (shape[0] - ((newshape[0] - 1) * ss[0]) - ws[0])
    logger.debug(f"Campioni rimanenti dopo l'ultima finestra completa: {remaining_samples}")
    
    if remaining_samples > 0:
        if remaining_samples > extreme_pad_samples:
            num_padding = ws[0] - remaining_samples
            padding_start = np.random.randint(1, num_padding) if num_padding > 1 else 1
            padding_end = num_padding - padding_start
            
            # Creo ultima finestra con padding per i dati
            last_window_data = np.concatenate((a[-remaining_samples:], np.zeros((padding_end, a.shape[1]))), axis=0)
            last_window_data = np.concatenate((np.zeros((padding_start, a.shape[1])), last_window_data), axis=0)
            
            # Creo ultima finestra con padding per gli indici
            last_window_indices = np.concatenate((original_row_indices[-remaining_samples:], np.full(padding_end, -1)))
            last_window_indices = np.concatenate((np.full(padding_start, -1), last_window_indices))
            
            # Aggiungo le ultime finestre
            strided = np.concatenate((strided, last_window_data[None, :, :]), axis=0)
            row_indices_strided = np.concatenate((row_indices_strided, last_window_indices[None, :]), axis=0)
            
            if remaining_samples < min_pad_samples and remaining_samples > extreme_pad_samples:
                padding_code_vector.append(2)
                logger.info(f"Padding estremo applicato. Padding code: 2")
            elif remaining_samples < ws[0]:
                padding_code_vector.append(1)  
                logger.info(f"Padding normale applicato. Padding code: 1")
            
            logger.debug(f"Nuova finestra con padding: {last_window_data.shape}")
        else:
            logger.info("Non è stato applicato padding perché i campioni rimanenti sono troppo pochi.")
    
    # Assicuriamo che padding_code_vector abbia la lunghezza corretta
    while len(padding_code_vector) < strided.shape[0]:
        padding_code_vector.append(0)  # 0 = nessun padding
    
    if not flatten:
        return strided, np.array(padding_code_vector), row_indices_strided
    
    # Flatten se richiesto
    meat = len(ws) if ws.shape else 0
    firstdim = (np.prod(newshape[:-meat]),) if ws.shape else ()
    dim = firstdim + tuple(newshape[-meat:])
    strided = strided.reshape(dim)
    
    logger.info(f"Numero totale di finestre (dopo padding finale): {strided.shape[0]}")
    return strided, np.array(padding_code_vector), row_indices_strided


def process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step):
    df = pd.read_csv(file_path)
    logger.debug(f"sto leggendo il file csv: {file_path}")

    kid_ids = df['kid_id'].unique()
    print(">>> Kid_ids:", kid_ids)
    X, Y = [], []
    kid_ids_all = []
    is_consecutive = []
    row_indices_all = []  # Nuova lista per tracciare gli indici delle righe
    
    kid_id_action_count = {}
    global_window_id = 0

    for kid_id in kid_ids:
        kid_mask = df['kid_id'] == kid_id
        kid_df = df[kid_mask].reset_index(drop=True)
        
        # Ottieni gli indici originali delle righe per questo bambino
        original_kid_indices = df.index[kid_mask].tolist()
        
        X_kid = df[df['kid_id'] == kid_id].drop(columns=['Timestamp', 'Accel_WR_X' ,'Accel_WR_Y' ,'Accel_WR_Z' ,'Date_time', 'action', 'action_id', 'G',
                              'communication', 'social_interaction', 'restricted_repetitive_behaviour',
                              'ados_total_score', 'I', 'E', 'toy_id','kid_id']).to_numpy()
        Y_kid = df[df['kid_id'] == kid_id]['action_id'].to_numpy()

        # Estraggo i timestamp per questo bambino
        timestamps_kid = df[df['kid_id'] == kid_id]['Timestamp'].to_numpy()
        logger.info(f"Kid_id: {kid_id}, X_kid shape: {X_kid.shape}, Y_kid shape: {Y_kid.shape}")

        # Applicazione della sliding window CON TRACKING DELLE RIGHE
        X_windows, padding_codes_x, row_indices_windows =sliding_window(
            X_kid, 
            ws=(sliding_window_length, X_kid.shape[1]), 
            ss=(sliding_window_step, X_kid.shape[1]), 
            min_pad_samples=30, 
            extreme_pad_samples=10,
            start_row_idx=original_kid_indices[0]  # Passa l'indice di partenza
        )
        
        logger.debug(f"X_windows shape after sliding window: {X_windows.shape}")
        logger.debug(f"Row indices windows shape: {row_indices_windows.shape}")
        logger.debug(f"Padding codes for X: {padding_codes_x}")

        # Sliding window per le labels
        Y_windows_full, padding_codes_y, _ = sliding_window(
            Y_kid.reshape(-1, 1), 
            ws=(sliding_window_length, 1), 
            ss=(sliding_window_step, 1), 
            min_pad_samples=30, 
            extreme_pad_samples=10,
            start_row_idx=original_kid_indices[0]
        )
        
        logger.debug(f"Y_windows_full shape after sliding window: {Y_windows_full.shape}")
        logger.debug(f"Padding codes for Y: {padding_codes_y}")
        
        # Estraggo l'ultima etichetta di ogni finestra
        Y_windows = np.asarray([window[np.nonzero(window)[0][0]] if np.any(window != 0) else 0 for window in Y_windows_full])
        logger.debug(f"Y_windows shape after extraction: {Y_windows.shape}")

        # Calcolo la consecutività per ogni finestra basata sui timestamp
        consecutivity_info = calculate_window_consecutivity(
            timestamps_kid, sliding_window_length, sliding_window_step, len(X_windows)
        )
        
        # Salvo le informazioni per ogni finestra
        for i in range(len(X_windows)):
            start_idx = i * sliding_window_step
            end_idx = min(start_idx + sliding_window_length, len(kid_df))
            
            is_consec = consecutivity_info[i]
            is_consecutive.append(is_consec)
            global_window_id += 1

        X.append(X_windows)
        Y.append(Y_windows)
        row_indices_all.append(row_indices_windows)  # Salva gli indici delle righe
        kid_id_action_count[int(kid_id)] = len(Y_windows)
        kid_ids_all.extend([kid_id] * len(Y_windows))

        # Log delle azioni per ogni kid
        for action_id in np.unique(Y_windows):
            action_count = len(Y_windows[Y_windows == action_id])
            logger.info(f"Kid_id: {kid_id}, Action_id: {action_id}, Action_count: {action_count}")
    
    logger.debug(f"Conteggio finale di finestre per ogni bambino: {kid_id_action_count}")
    
    # Concateno tutto
    X_all = np.concatenate(X, axis=0)
    Y_all = np.concatenate(Y, axis=0)
    row_indices_all_concat = np.concatenate(row_indices_all, axis=0)  # Concatena gli indici
    kid_ids_all = np.array(kid_ids_all)
    is_consecutive = np.array(is_consecutive)

    return X_all, Y_all, kid_id_action_count, is_consecutive, row_indices_all_concat


def calculate_window_consecutivity(timestamps, window_length, window_step, num_windows):
    """
    Calcola se ogni finestra contiene timestamp consecutivi.
    
    Args:
        timestamps: array dei timestamp ordinati
        window_length: lunghezza della finestra
        window_step: passo della finestra
        num_windows: numero di finestre generate
    
    Returns:
        list: lista di booleani che indica se ogni finestra è consecutiva
    """
    consecutivity = [] #lista per tenere traccia della consecutività delle finestre
    
    # Calcolo la differenza media tra timestamp consecutivi => la differenza tra ogni coppia consecutiva di timestamp (esempio: [1, 2, 3] => [1, 1])
    time_diffs = np.diff(timestamps)
    median_diff = np.median(time_diffs) #calcolo la mediana delle differenze tra timestamp consecutivi (stima della differenza tipica tra timestamp consecutivi)
    
    # Soglia per considerare un gap troppo grande (es. 3 volte la differenza mediana), se la differenza tra due timestamp è maggiore di 3 volte la mediana, viene considerato un "buco" anomalo (cioè non consecutivo).
    gap_threshold = median_diff * 3
    
    #itero su tutte le finestre
    for i in range(num_windows):
        start_idx = i * window_step # Calcolo l'indice di inizio della finestra
        end_idx = min(start_idx + window_length, len(timestamps)) # Calcolo l'indice di fine della finestra
        
        if end_idx <= start_idx + 1:
            # Finestra troppo piccola (ma non capita mai)
            consecutivity.append(False)
            continue
            
        # Estraggo i timestamp della finestra
        window_timestamps = timestamps[start_idx:end_idx]
        
        # Calcolo le differenze tra timestamp consecutivi nella finestra
        window_diffs = np.diff(window_timestamps)
        
        # Verifico se ci sono gap troppo grandi rispetto alla soglia calcolata
        has_large_gaps = np.any(window_diffs > gap_threshold)
        
        # La finestra è consecutiva se non ha gap grandi
        is_consec = not has_large_gaps #Se non ci sono grandi gap, is_consec è True (finestra consecutiva)
        consecutivity.append(is_consec) #Restituisce la lista di booleani, uno per ogni finestra, che indica se i timestamp nella finestra erano consecutivi (True) o no (False)
        
    return consecutivity


def norm_shape(shape):
    '''
    Normalizza le forme degli array numpy, assicurandosi che siano sempre espresse come tuple.
    Anche se viene passato un numero intero, verrà restituita una tupla con un solo elemento.

    Parametri:
        shape - Può essere un intero oppure una tupla (o lista) di interi.

    Ritorna:
        Una tupla contenente la forma normalizzata.
    '''
    try:
        # Provo a convertire il valore in un intero
        i = int(shape)
        return (i,)  # Se è un intero, lo restituisco come una tupla di un solo elemento
    except TypeError:
        # Se il valore passato non è un numero, ignoro l'errore e passo al tentativo successivo
        pass
    
    try:
        # Provo a convertire il valore in una tupla
        t = tuple(shape)
        return t  # Se l'operazione ha successo, restituisco la tupla risultante
    except TypeError:
        # Se il valore passato non è una sequenza iterabile, ignoro l'errore e procedo
        pass
    
    # Se nessuno dei due tentativi ha avuto successo, sollevo un errore
    raise TypeError('shape deve essere un intero o una tupla di interi')


In [ ]:

import numpy as np
from numpy.lib.stride_tricks import as_strided as ast
from utils.log_config import logger
# applico sliding window con la funzione process_csv
nb_sensor_channels = 9
sliding_window_length = 100
sliding_window_step = 50

# ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al giocattolo spoon
# e poi concateno tutto in un unica x e y 

X, Y = [], []
kid_action_counts = {}
all_consecutivity = []
all_window_indices = []
all_row_indices = []  # NUOVO: per tracciare gli indici delle righe
global_window_id = 0

for action_file in [f for f in os.listdir(path) if f.endswith('.csv') and f.startswith('df_spoon_action_') and not f.endswith('normalized_pt.csv')]:
    file_path = os.path.join(path, action_file)
    action = action_file.split('_')[-1].split('.')[0]

    # MODIFICATO: ora la funzione restituisce anche row_indices
    X_windows, Y_windows, kid_id_action_dict, is_consecutive, row_indices = process_csv(
        file_path, nb_sensor_channels, sliding_window_length, sliding_window_step
    )

    # Aggiungo informazioni aggiuntive per ogni finestra
    for i in range(len(X_windows)):
        all_window_indices.append({
            'global_window_id': global_window_id,
            'action_id': action,
            'file': action_file,  # Utile per debug
            'local_window_id': i   # Indice locale nella singola azione
        })
        global_window_id += 1
    
    X.append(X_windows)
    Y.append(Y_windows)
    all_consecutivity.extend(is_consecutive)
    all_row_indices.append(row_indices)  # NUOVO: salvo gli indici delle righe

    logger.info(f"Numero totale di finestre per l'azione {action}: {len(X_windows)}")
    logger.info(f"Finestre consecutive per l'azione {action}: {sum(is_consecutive)}")
    logger.info(f"Finestre non consecutive per l'azione {action}: {len(is_consecutive) - sum(is_consecutive)}")
    logger.info(f"Numero totale di finestre per l'azione {action}: {len(X_windows)}")
    
    # Log aggiuntivo per il tracking delle righe
    logger.info(f"Shape degli indici delle righe per l'azione {action}: {row_indices.shape}")
    
    kid_action_counts[f"spoon_action_{action}"] = kid_id_action_dict
    logger.info(f"Contenuto finale di kid_action_counts: {kid_action_counts}")

# Concateno tutti i dati in un unico array per X, Y e gli indici delle righe
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)
all_consecutivity = np.array(all_consecutivity)
all_row_indices = np.concatenate(all_row_indices, axis=0)  # NUOVO: concateno gli indici delle righe

logger.info(f"Dimensioni di X dopo concatenazione: {X.shape}")
logger.info(f"Dimensioni di Y dopo concatenazione: {Y.shape}")
logger.info(f"Dimensioni degli indici delle righe dopo concatenazione: {all_row_indices.shape}")

# NUOVO: Funzioni di utilità per analizzare i dati
def analyze_padding_statistics():
    """Analizza le statistiche del padding applicato"""
    total_windows = len(all_row_indices)
    windows_with_padding = 0
    total_padding_samples = 0
    
    for window_idx, window_rows in enumerate(all_row_indices):
        padding_count = np.sum(window_rows == -1)
        if padding_count > 0:
            windows_with_padding += 1
            total_padding_samples += padding_count
            
    logger.info(f"Statistiche padding:")
    logger.info(f"  - Finestre totali: {total_windows}")
    logger.info(f"  - Finestre con padding: {windows_with_padding}")
    logger.info(f"  - Percentuale finestre con padding: {windows_with_padding/total_windows*100:.2f}%")
    logger.info(f"  - Campioni di padding totali: {total_padding_samples}")
    logger.info(f"  - Media campioni padding per finestra: {total_padding_samples/total_windows:.2f}")

def get_original_dataframe_rows(window_index):
    """
    Restituisce gli indici delle righe originali del dataframe per una specifica finestra
    
    Args:
        window_index: Indice della finestra (0-based)
    
    Returns:
        dict: Dizionario con informazioni sulla finestra
    """
    if window_index >= len(all_row_indices):
        raise ValueError(f"Indice finestra {window_index} fuori range. Max: {len(all_row_indices)-1}")
    
    window_rows = all_row_indices[window_index]
    real_rows = window_rows[window_rows != -1]  # Righe reali (non padding)
    padding_positions = np.where(window_rows == -1)[0]  # Posizioni del padding
    
    return {
        'window_index': window_index,
        'total_samples': len(window_rows),
        'real_samples': len(real_rows),
        'padding_samples': len(padding_positions),
        'original_dataframe_rows': real_rows.tolist(),
        'padding_positions': padding_positions.tolist(),
        'action': all_window_indices[window_index]['action_id'],
        'file': all_window_indices[window_index]['file']
    }

def find_windows_with_specific_row(dataframe_row_index):
    """
    Trova tutte le finestre che contengono una specifica riga del dataframe originale
    
    Args:
        dataframe_row_index: Indice della riga nel dataframe originale
    
    Returns:
        list: Lista degli indici delle finestre che contengono quella riga
    """
    windows_containing_row = []
    
    for window_idx, window_rows in enumerate(all_row_indices):
        if dataframe_row_index in window_rows:
            windows_containing_row.append({
                'window_index': window_idx,
                'position_in_window': np.where(window_rows == dataframe_row_index)[0][0],
                'action': all_window_indices[window_idx]['action_id']
            })
    
    return windows_containing_row

# Esegui analisi del padding
analyze_padding_statistics()

# ESEMPI DI USO:
# 1. Vedere le informazioni per la finestra 0
# window_info = get_original_dataframe_rows(0)
# print("Info finestra 0:", window_info)

# 2. Trovare tutte le finestre che contengono la riga 100 del dataframe
# windows_with_row_100 = find_windows_with_specific_row(100)
# print("Finestre contenenti riga 100:", windows_with_row_100)

# 3. Controllare se una finestra specifica ha padding
# window_5_rows = all_row_indices[5]
# has_padding = np.any(window_5_rows == -1)
# padding_count = np.sum(window_5_rows == -1)
# print(f"Finestra 5 ha padding: {has_padding}, numero campioni padding: {padding_count}")

logger.info("Setup completato con tracking delle righe del dataframe originale!")

2025-07-27 15:08:05,703 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\downstream_data\df_spoon_action_10.csv
2025-07-27 15:08:05,706 - INFO - myapp - Kid_id: 3002, X_kid shape: (657, 9), Y_kid shape: (657,)


>>> Kid_ids: [3002]


NameError: name 'ast' is not defined